In [1]:
import mysql.connector
import time
import logging
from datetime import datetime, timezone
from getpass import getpass

# ---------------------------------------------------------
# ENTERPRISE LOGGING
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ---------------------------------------------------------
# DB CONNECTION (SECURE)
# ---------------------------------------------------------
db = mysql.connector.connect(
    host="localhost",
    user="Oyeniyi_ETL",
    password=getpass("Enter MySQL password: "),
    database="electricity_capstone",
    autocommit=False
)
cur = db.cursor()

batch_id    = int(time.time())
pipeline_id = "etl_v1"
run_ts      = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

logging.info(f"PIPELINE START | batch_id={batch_id} | pipeline_id={pipeline_id}")

# ---------------------------------------------------------
# LOAD DIMENSIONS INTO MEMORY
# ---------------------------------------------------------
logging.info("Loading dim_time...")
cur.execute("SELECT datetime_utc, time_id FROM dim_time")
time_map = {dt.strftime("%Y-%m-%d %H:%M:%S"): tid for dt, tid in cur.fetchall()}
logging.info(f"dim_time loaded: {len(time_map):,} keys")

logging.info("Loading dim_country...")
cur.execute("SELECT country_code, country_id FROM dim_country")
code_map = {(cc or '').strip().upper(): cid for cc, cid in cur.fetchall()}
logging.info(f"dim_country loaded: {len(code_map):,} ISO2 codes")

cur.execute("SELECT country_name, country_id FROM dim_country")
name_map = {(cn or '').strip(): cid for cn, cid in cur.fetchall()}

# ---------------------------------------------------------
# ROW COUNTS
# ---------------------------------------------------------
cur.execute("SELECT COUNT(*) FROM tmp_cleaned_demand")
demand_total = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM tmp_cleaned_price")
price_total = cur.fetchone()[0]

logging.info(f"Rows to load → demand={demand_total:,} | price={price_total:,}")

CHUNK = 200_000

# ---------------------------------------------------------
# ENTERPRISE PROGRESS TRACKER
# ---------------------------------------------------------
def progress(label, processed, total, inserted, skipped, start_ts):
    elapsed = time.time() - start_ts
    pct = (processed / total * 100) if total else 0
    rate = processed / elapsed if elapsed > 0 else 0
    eta = (total - processed) / rate if rate > 0 else 0

    logging.info(
        f"{label} | {processed:,}/{total:,} ({pct:5.1f}%) "
        f"inserted={inserted:,} skipped={skipped:,} "
        f"rate={rate:,.0f} rows/s elapsed={elapsed:,.1f}s eta={eta:,.1f}s"
    )

# ---------------------------------------------------------
# FACT DEMAND LOADER (TOP 0.1%)
# ---------------------------------------------------------
def load_fact_demand():
    logging.info("Starting fact_demand load...")
    offset = 0
    processed = inserted = skipped = 0
    start_ts = time.time()

    while True:
        cur.execute(f"""
            SELECT datetime_utc, country_code, cov_ratio, value, value_scaled
            FROM tmp_cleaned_demand
            ORDER BY datetime_utc
            LIMIT {CHUNK} OFFSET {offset}
        """)
        rows = cur.fetchall()
        if not rows:
            break

        batch = []
        for dt_utc, iso2, cov, val, val_scaled in rows:
            processed += 1

            dt_key = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
            iso_key = (iso2 or '').strip().upper()

            time_id = time_map.get(dt_key)
            country_id = code_map.get(iso_key)

            if not time_id or not country_id:
                skipped += 1
                continue

            batch.append((
                time_id, country_id, cov, val, val_scaled,
                batch_id, pipeline_id, run_ts
            ))

        if batch:
            cur.executemany("""
                INSERT IGNORE INTO fact_demand (
                    time_id, country_id, cov_ratio, value, value_scaled,
                    batch_id, pipeline_id, load_ts
                )
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            """, batch)
            db.commit()
            inserted += cur.rowcount

        if processed % 50_000 == 0:
            progress("fact_demand", processed, demand_total, inserted, skipped, start_ts)

        offset += CHUNK

    progress("fact_demand FINAL", processed, demand_total, inserted, skipped, start_ts)

# ---------------------------------------------------------
# FACT PRICE LOADER (TOP 0.1%)
# ---------------------------------------------------------
def load_fact_price():
    logging.info("Starting fact_price load...")
    offset = 0
    processed = inserted = skipped = 0
    start_ts = time.time()

    while True:
        cur.execute(f"""
            SELECT datetime_utc, country_name, price_eur_mwhe
            FROM tmp_cleaned_price
            ORDER BY datetime_utc
            LIMIT {CHUNK} OFFSET {offset}
        """)
        rows = cur.fetchall()
        if not rows:
            break

        batch = []
        for dt_utc, cname, price in rows:
            processed += 1

            dt_key = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
            cname_key = (cname or '').strip()

            time_id = time_map.get(dt_key)
            country_id = name_map.get(cname_key)

            if not time_id or not country_id:
                skipped += 1
                continue

            batch.append((
                time_id, country_id, price,
                batch_id, pipeline_id, run_ts
            ))

        if batch:
            cur.executemany("""
                INSERT IGNORE INTO fact_price (
                    time_id, country_id, price_eur_mwhe,
                    batch_id, pipeline_id, load_ts
                )
                VALUES (%s,%s,%s,%s,%s,%s)
            """, batch)
            db.commit()
            inserted += cur.rowcount

        if processed % 50_000 == 0:
            progress("fact_price", processed, price_total, inserted, skipped, start_ts)

        offset += CHUNK

    progress("fact_price FINAL", processed, price_total, inserted, skipped, start_ts)

# ---------------------------------------------------------
# RUN PIPELINE
# ---------------------------------------------------------
load_fact_demand()
load_fact_price()

logging.info("PIPELINE COMPLETE.")
cur.close()
db.close()


Enter MySQL password:  ········


2026-04-25 19:55:39,725 [INFO] PIPELINE START | batch_id=1777143339 | pipeline_id=etl_v1
2026-04-25 19:55:39,742 [INFO] Loading dim_time...
2026-04-25 19:55:45,488 [INFO] dim_time loaded: 94,224 keys
2026-04-25 19:55:45,493 [INFO] Loading dim_country...
2026-04-25 19:55:45,526 [INFO] dim_country loaded: 32 ISO2 codes
2026-04-25 19:56:29,119 [INFO] Rows to load → demand=2,071,728 | price=2,209,368
2026-04-25 19:56:29,321 [INFO] Starting fact_demand load...
2026-04-25 19:57:36,119 [INFO] fact_demand | 200,000/2,071,728 (  9.7%) inserted=171,799 skipped=28,173 rate=2,995 rows/s elapsed=66.8s eta=624.9s
2026-04-25 19:58:19,193 [INFO] fact_demand | 400,000/2,071,728 ( 19.3%) inserted=340,666 skipped=59,277 rate=3,641 rows/s elapsed=109.9s eta=459.1s
2026-04-25 19:59:44,632 [INFO] fact_demand | 600,000/2,071,728 ( 29.0%) inserted=506,777 skipped=93,166 rate=3,073 rows/s elapsed=195.3s eta=479.0s
2026-04-25 20:01:15,644 [INFO] fact_demand | 800,000/2,071,728 ( 38.6%) inserted=673,815 skipped=